In [1]:
!pip install sentence-transformers
import os
import json
import zipfile
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

# STEP 1: EXTRACTION
zip_path = '/content/data/processed_data.zip'
extract_path = '/content/data/'
os.makedirs('/content/data/embeddings/', exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete. Folder structure:")
!find /content/data/processed_data -maxdepth 2 -not -path '*/.*'

Extraction complete. Folder structure:
/content/data/processed_data
/content/data/processed_data/family
/content/data/processed_data/family/child_custody.jsonl
/content/data/processed_data/family/family.jsonl
/content/data/processed_data/family/maintenance.jsonl
/content/data/processed_data/family/divorce.jsonl
/content/data/processed_data/labour
/content/data/processed_data/labour/wrongful_termination.jsonl
/content/data/processed_data/consumer
/content/data/processed_data/consumer/service_deficiency.jsonl
/content/data/processed_data/consumer/consumer.jsonl
/content/data/processed_data/consumer/online_fraud.jsonl
/content/data/processed_data/criminal
/content/data/processed_data/criminal/FIR_procedure.jsonl
/content/data/processed_data/criminal/domestic_violence.jsonl
/content/data/processed_data/criminal/bail_general.jsonl
/content/data/processed_data/criminal/drug_offenses.jsonl
/content/data/processed_data/criminal/assault_violence.jsonl
/content/data/processed_data/criminal/crimi

In [2]:
# STEP 2: FILE DISCOVERY
file_registry = []
total_rows_to_process = 0

root_dir = '/content/data/processed_data'
for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.jsonl'):
            full_path = os.path.join(root, file)
            domain = os.path.basename(root)
            subdomain = file.replace('.jsonl', '')

            # Quick count for estimation
            count = sum(1 for _ in open(full_path, 'r'))
            file_registry.append({
                'path': full_path,
                'domain': domain,
                'subdomain': subdomain,
                'row_count': count
            })
            total_rows_to_process += count

print(f"Discovered {len(file_registry)} files across domains.")
for f in file_registry:
    print(f"- {f['domain']}/{f['subdomain']}.jsonl: {f['row_count']} rows")

Discovered 20 files across domains.
- family/child_custody.jsonl: 508 rows
- family/family.jsonl: 202 rows
- family/maintenance.jsonl: 322 rows
- family/divorce.jsonl: 715 rows
- labour/wrongful_termination.jsonl: 752 rows
- consumer/service_deficiency.jsonl: 725 rows
- consumer/consumer.jsonl: 287 rows
- consumer/online_fraud.jsonl: 518 rows
- criminal/FIR_procedure.jsonl: 716 rows
- criminal/domestic_violence.jsonl: 709 rows
- criminal/bail_general.jsonl: 588 rows
- criminal/drug_offenses.jsonl: 615 rows
- criminal/assault_violence.jsonl: 876 rows
- criminal/criminal.jsonl: 5927 rows
- criminal/fraud_cheating.jsonl: 56 rows
- general/unclassified2.jsonl: 483 rows
- general/unclassified.jsonl: 643 rows
- constitutional/constitutional.jsonl: 252 rows
- constitutional/illegal_detention.jsonl: 570 rows
- motor_accident/compensation.jsonl: 745 rows


In [3]:
# STEP 3 & 4: EMBEDDING LOGIC & SAVING
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Model loaded on: {device}")

summary_data = []
start_time = time.time()
rows_processed_total = 0

def process_batch(batch_texts, batch_metadata, embeddings_list, metadata_list):
    if not batch_texts:
        return
    embs = model.encode(
        batch_texts,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    embeddings_list.append(embs)
    metadata_list.extend(batch_metadata)

for file_info in file_registry:
    f_path = file_info['path']
    domain = file_info['domain']
    subdomain = file_info['subdomain']

    file_embeddings = []
    file_metadata = []
    batch_texts = []
    batch_meta = []
    skipped = 0
    processed_in_file = 0

    with open(f_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            text = data.get('text', '')

            if not text or len(text) < 30:
                skipped += 1
                continue

            # Metadata extraction
            meta = {"domain": domain, "subdomain": subdomain}
            for field in ['id', 'case_name', 'court', 'year', 'legal_issue', 'user_intent', 'stage', 'authority_score', 'source']:
                meta[field] = data.get(field)

            # Join lists
            meta['sections'] = "|".join(data.get('sections', [])) if isinstance(data.get('sections'), list) else data.get('sections')
            meta['acts'] = "|".join(data.get('acts', [])) if isinstance(data.get('acts'), list) else data.get('acts')

            batch_texts.append(text)
            batch_meta.append(meta)

            if len(batch_texts) == 64:
                process_batch(batch_texts, batch_meta, file_embeddings, file_metadata)
                processed_in_file += 64
                batch_texts, batch_meta = [], []

                if processed_in_file % 128 == 0 or processed_in_file == 64:
                    print(f"[{domain}/{subdomain}.jsonl] Processed {processed_in_file}/{file_info['row_count']} rows...")

    # Final batch
    if batch_texts:
        process_batch(batch_texts, batch_meta, file_embeddings, file_metadata)
        processed_in_file += len(batch_texts)

    # Concatenate and Save
    if file_embeddings:
        final_embeddings = np.vstack(file_embeddings)
        save_name_emb = f"/content/data/embeddings/embeddings_{domain}_{subdomain}.npy"
        save_name_meta = f"/content/data/embeddings/metadata_{domain}_{subdomain}.json"

        np.save(save_name_emb, final_embeddings)
        with open(save_name_meta, 'w') as jf:
            json.dump(file_metadata, jf)

        print(f"✅ {domain}/{subdomain}.jsonl → {processed_in_file} rows | Shape: {final_embeddings.shape} | Saved.")
        summary_data.append([domain, subdomain, processed_in_file, skipped, final_embeddings.shape])

    # Time Estimation after first file
    rows_processed_total += processed_in_file
    if len(summary_data) == 1:
        elapsed = time.time() - start_time
        rps = rows_processed_total / elapsed
        remaining_rows = total_rows_to_process - rows_processed_total
        est_min = (remaining_rows / rps) / 60
        print(f"\nEstimated time remaining: {est_min:.2f} minutes\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded on: cpu
[family/child_custody.jsonl] Processed 64/508 rows...
[family/child_custody.jsonl] Processed 128/508 rows...
[family/child_custody.jsonl] Processed 256/508 rows...
[family/child_custody.jsonl] Processed 384/508 rows...
✅ family/child_custody.jsonl → 508 rows | Shape: (508, 384) | Saved.

Estimated time remaining: 34.96 minutes

[family/family.jsonl] Processed 64/202 rows...
[family/family.jsonl] Processed 128/202 rows...
✅ family/family.jsonl → 202 rows | Shape: (202, 384) | Saved.
[family/maintenance.jsonl] Processed 64/322 rows...
[family/maintenance.jsonl] Processed 128/322 rows...
[family/maintenance.jsonl] Processed 256/322 rows...
✅ family/maintenance.jsonl → 322 rows | Shape: (322, 384) | Saved.
[family/divorce.jsonl] Processed 64/715 rows...
[family/divorce.jsonl] Processed 128/715 rows...
[family/divorce.jsonl] Processed 256/715 rows...
[family/divorce.jsonl] Processed 384/715 rows...
[family/divorce.jsonl] Processed 512/715 rows...
[family/divorce.jsonl] 

In [4]:
# STEP 5 & 6: SUMMARY & VALIDATION
print("\n" + "="*50)
print(f"{'Domain':<15} | {'File':<25} | {'Rows':<6} | {'Skipped':<7} | {'Shape'}")
print("-"*85)
for row in summary_data:
    print(f"{row[0]:<15} | {row[1]:<25} | {row[2]:<6} | {row[3]:<7} | {row[4]}")

print("\nRunning Final Validation...")
import glob
all_np_files = glob.glob("/content/data/embeddings/embeddings_*.npy")

for np_file in all_np_files:
    meta_file = np_file.replace('embeddings_', 'metadata_').replace('.npy', '.json')
    emb = np.load(np_file)
    with open(meta_file, 'r') as f:
        meta = json.load(f)

    check_dim = emb.shape[1] == 384
    check_count = emb.shape[0] == len(meta)
    status = "PASS" if check_dim and check_count else "FAIL"
    print(f"[{os.path.basename(np_file)}] Dim: {emb.shape[1]} | Rows: {emb.shape[0]} | Metadata: {len(meta)} -> {status}")

# Export to /content for easy download
!cp -r /content/data/embeddings /content/nyayasetu_embeddings
print("\nAll files copied to /content/nyayasetu_embeddings for export.")


Domain          | File                      | Rows   | Skipped | Shape
-------------------------------------------------------------------------------------
family          | child_custody             | 508    | 0       | (508, 384)
family          | family                    | 202    | 0       | (202, 384)
family          | maintenance               | 322    | 0       | (322, 384)
family          | divorce                   | 715    | 0       | (715, 384)
labour          | wrongful_termination      | 752    | 0       | (752, 384)
consumer        | service_deficiency        | 725    | 0       | (725, 384)
consumer        | consumer                  | 287    | 0       | (287, 384)
consumer        | online_fraud              | 518    | 0       | (518, 384)
criminal        | FIR_procedure             | 716    | 0       | (716, 384)
criminal        | domestic_violence         | 709    | 0       | (709, 384)
criminal        | bail_general              | 588    | 0       | (588, 384)
crimin

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import shutil

# Define paths
source_folder = '/content/nyayasetu_embeddings'
drive_dest_folder = '/content/drive/My Drive/Hackathon'

# Create destination if it doesn't exist
os.makedirs(drive_dest_folder, exist_ok=True)

# Copy the contents
try:
    # Using cp -r via shell for efficiency with many small files
    !cp -r "{source_folder}" "{drive_dest_folder}"
    print(f"\u2705 Successfully exported embeddings to: {drive_dest_folder}")
except Exception as e:
    print(f"\u274c Error during export: {e}")

✅ Successfully exported embeddings to: /content/drive/My Drive/Hackathon
